# Synthesis Benchmark: Multi-Method Comparison with Explainable AI

**Crop AI** — Six-method Before/After augmentation analysis with LIME & SHAP

Covers:
1. Data augmentation: multivariate normal (3x) vs SMOTE
2. Six ML methods: RF, GB, Stacking, AdaBoost, SVM, MLP
3. Before-augmentation vs after-augmentation accuracy table
4. Explainable AI: LIME + SHAP for rice prediction
5. LLM pipeline architecture diagram

In [1]:
!pip install imbalanced-learn lime shap -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from collections import OrderedDict
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Colour palette — matches model_benchmark.py exactly
P = {
    "bg":      "#0a0a0a",
    "panel":   "#111111",
    "border":  "#1e1e1e",
    "text":    "#e0e0e0",
    "sub":     "#888888",
    "green":   "#22c55e",
    "blue":    "#3b82f6",
    "amber":   "#f59e0b",
    "red":     "#ef4444",
    "purple":  "#a855f7",
}

FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
OUT_DIR  = Path('.')  # save PNGs alongside notebook

print('Imports complete')

Imports complete


---
## Section 1 — Setup & Data

In [3]:
print('Loading Kaggle crop recommendation dataset...')
raw_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'atharvaingle/crop-recommendation-dataset',
    'Crop_recommendation.csv'
)

X_raw = raw_df[FEATURES].values
y_raw = raw_df['label'].values   # string labels already

print(f'Dataset loaded: X_raw {X_raw.shape}, y_raw {y_raw.shape}')
print(f'Classes ({len(np.unique(y_raw))}): {sorted(np.unique(y_raw))}')
print(f'Samples per class: {pd.Series(y_raw).value_counts().iloc[0]} (uniform)')
print(raw_df.head())

Loading Kaggle crop recommendation dataset...


Dataset loaded: X_raw (2200, 7), y_raw (2200,)
Classes (22): ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']
Samples per class: 100 (uniform)
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice


In [4]:
# ── Multivariate normal augmentation (copied verbatim from CROP_RECOMMENDER.ipynb Cell 3) ──
print('Generating synthetic data via multivariate normal augmentation...')

label_col         = 'label'
feature_cols      = FEATURES
augmentation_factor = 3

synthetic_samples = []
for crop_label in raw_df[label_col].unique():
    crop_data = raw_df[raw_df[label_col] == crop_label][feature_cols].values
    mean      = crop_data.mean(axis=0)
    cov       = np.cov(crop_data.T)
    n_synthetic = len(crop_data) * (augmentation_factor - 1)
    synthetic_crop_data = np.random.multivariate_normal(mean, cov, int(n_synthetic))
    synthetic_crop_data = np.clip(synthetic_crop_data,
                                   raw_df[feature_cols].min().values,
                                   raw_df[feature_cols].max().values)
    for sample in synthetic_crop_data:
        synthetic_samples.append(list(sample) + [crop_label])

synthetic_df = pd.DataFrame(synthetic_samples, columns=feature_cols + [label_col])
combined_df  = pd.concat([raw_df, synthetic_df], ignore_index=True)
combined_df  = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

X_aug = combined_df[FEATURES].values
y_aug = combined_df['label'].values

print(f'Original : {X_raw.shape[0]} samples')
print(f'Augmented: {X_aug.shape[0]} samples ({augmentation_factor}x)')
print(f'X_aug shape: {X_aug.shape}')

Generating synthetic data via multivariate normal augmentation...
Original : 2200 samples
Augmented: 6600 samples (3x)
X_aug shape: (6600, 7)


In [5]:
from imblearn.over_sampling import SMOTE

print('Applying SMOTE augmentation (comparison baseline)...')
le         = LabelEncoder()
y_raw_enc  = le.fit_transform(y_raw)

print('Class distribution BEFORE SMOTE:')
for cls, cnt in zip(*np.unique(y_raw_enc, return_counts=True)):
    print(f'  {le.classes_[cls]:<15}: {cnt}')

smote = SMOTE(random_state=42)
X_smote, y_smote_enc = smote.fit_resample(X_raw, y_raw_enc)
y_smote = le.inverse_transform(y_smote_enc)

print(f'\nClass distribution AFTER SMOTE:')
for cls, cnt in zip(*np.unique(y_smote_enc, return_counts=True)):
    print(f'  {le.classes_[cls]:<15}: {cnt}')

n_new = X_smote.shape[0] - X_raw.shape[0]
print(f'\nSMOTE result : {X_smote.shape[0]} samples (was {X_raw.shape[0]})')
print(f'New samples added: {n_new}')
print()
print('NOTE: SMOTE added 0 new samples. This is the correct and expected result.')
print('The Kaggle crop dataset is perfectly balanced (100 samples/class for all 22 classes).')
print('SMOTE only adds synthetic samples to minority classes; on balanced data it is a no-op.')
print('This is an honest benchmark result, not a failure.')

Applying SMOTE augmentation (comparison baseline)...
Class distribution BEFORE SMOTE:
  apple          : 100
  banana         : 100
  blackgram      : 100
  chickpea       : 100
  coconut        : 100
  coffee         : 100
  cotton         : 100
  grapes         : 100
  jute           : 100
  kidneybeans    : 100
  lentil         : 100
  maize          : 100
  mango          : 100
  mothbeans      : 100
  mungbean       : 100
  muskmelon      : 100
  orange         : 100
  papaya         : 100
  pigeonpeas     : 100
  pomegranate    : 100
  rice           : 100
  watermelon     : 100

Class distribution AFTER SMOTE:
  apple          : 100
  banana         : 100
  blackgram      : 100
  chickpea       : 100
  coconut        : 100
  coffee         : 100
  cotton         : 100
  grapes         : 100
  jute           : 100
  kidneybeans    : 100
  lentil         : 100
  maize          : 100
  mango          : 100
  mothbeans      : 100
  mungbean       : 100
  muskmelon      : 100
  orang

In [6]:
# ── Train/test splits ────────────────────────────────────────────────────────
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42)

X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_aug, y_aug, test_size=0.2, stratify=y_aug, random_state=42)

print(f'Raw    — train: {X_tr_r.shape[0]}, test: {X_te_r.shape[0]}')
print(f'Aug    — train: {X_tr_a.shape[0]}, test: {X_te_a.shape[0]}')

# ── Shared pipeline factory ───────────────────────────────────────────────────
def make_pipeline(clf):
    return Pipeline([
        ('mm',  MinMaxScaler()),
        ('std', StandardScaler()),
        ('clf', clf),
    ])

print('make_pipeline helper defined')

Raw    — train: 1760, test: 440
Aug    — train: 5280, test: 1320
make_pipeline helper defined


In [7]:
# Extract one rice sample from the raw test set for XAI
rice_idx    = np.where(y_te_r == 'rice')[0][0]
rice_sample = X_te_r[rice_idx]

print(f'Rice sample index : {rice_idx}')
print(f'Rice sample values:')
for fname, val in zip(FEATURES, rice_sample):
    print(f'  {fname:<12}: {val:.4f}')

Rice sample index : 6
Rice sample values:
  N           : 60.0000
  P           : 49.0000
  K           : 44.0000
  temperature : 20.7758
  humidity    : 84.4977
  ph          : 6.2448
  rainfall    : 240.0811


---
## Section 2 — Model Training Loop

Six classifiers trained with identical preprocessing pipelines (MinMaxScaler → StandardScaler → clf)  
under two conditions: **Before augmentation** (2,200 raw samples) and **After augmentation** (6,600 synthetic samples).

In [8]:
# Base estimators reused inside the Stacking classifier
RF50 = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
GB50 = GradientBoostingClassifier(n_estimators=50, random_state=42)

METHODS = OrderedDict([
    ('Random Forest', RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1)),

    ('Gradient Boost', GradientBoostingClassifier(
        n_estimators=100, random_state=42)),

    ('Stacking Ensemble', StackingClassifier(
        estimators=[('rf', RF50), ('gb', GB50), ('svm', SVC(probability=True))],
        final_estimator=LogisticRegression(max_iter=500),
        cv=3, n_jobs=-1)),

    ('AdaBoost', AdaBoostClassifier(
        n_estimators=100, random_state=42)),

    ('SVM (RBF)', SVC(
        kernel='rbf', probability=True, random_state=42)),

    ('MLP', MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)),
])

print('Methods defined:')
for name in METHODS:
    print(f'  {name}')

Methods defined:
  Random Forest
  Gradient Boost
  Stacking Ensemble
  AdaBoost
  SVM (RBF)
  MLP


In [9]:
splits = [
    ('before', X_tr_r, X_te_r, y_tr_r, y_te_r),
    ('after',  X_tr_a, X_te_a, y_tr_a, y_te_a),
]

results = {}  # {method_name: {'before': {...}, 'after': {...}}}

for name, clf_template in METHODS.items():
    results[name] = {}
    if name == 'Stacking Ensemble':
        print(f'Training Stacked Ensemble... (may take 2-5 min)')
    for label, Xtr, Xte, ytr, yte in splits:
        pipe = make_pipeline(clone(clf_template))
        pipe.fit(Xtr, ytr)
        y_pred = pipe.predict(Xte)
        results[name][label] = {
            'accuracy' : accuracy_score(yte, y_pred),
            'precision': precision_score(yte, y_pred, average='macro', zero_division=0),
            'recall'   : recall_score(yte, y_pred, average='macro', zero_division=0),
            'f1'       : f1_score(yte, y_pred, average='macro', zero_division=0),
            'pipeline' : pipe,
        }
        print(f'  {name:<22} [{label:6s}]  Acc={results[name][label]["accuracy"]:.4f}')

print('\nAll models trained.')

  Random Forest          [before]  Acc=0.9955


  Random Forest          [after ]  Acc=0.9879


  Gradient Boost         [before]  Acc=0.9886


  Gradient Boost         [after ]  Acc=0.9826
Training Stacked Ensemble... (may take 2-5 min)


  Stacking Ensemble      [before]  Acc=0.9955


  Stacking Ensemble      [after ]  Acc=0.9902
  AdaBoost               [before]  Acc=0.3136


  AdaBoost               [after ]  Acc=0.1765
  SVM (RBF)              [before]  Acc=0.9841


  SVM (RBF)              [after ]  Acc=0.9871


  MLP                    [before]  Acc=0.9909


  MLP                    [after ]  Acc=0.9909

All models trained.


---
## Section 3 — Synthesis Table

In [10]:
# ── Build results DataFrame ───────────────────────────────────────────────────
metrics      = ['accuracy', 'precision', 'recall', 'f1']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
method_names = list(METHODS.keys())

rows = []
for mname, mlabel in zip(metrics, metric_names):
    row = {'Metric': mlabel}
    for name in method_names:
        row[f'{name}\nBefore'] = round(results[name]['before'][mname], 4)
        row[f'{name}\nAfter']  = round(results[name]['after'][mname],  4)
    rows.append(row)

df_results = pd.DataFrame(rows).set_index('Metric')
print(df_results.to_string())

# Convenience references to RF augmented pipeline (used in XAI cells)
rf_aug_pipeline = results['Random Forest']['after']['pipeline']

           Random Forest\nBefore  Random Forest\nAfter  Gradient Boost\nBefore  Gradient Boost\nAfter  Stacking Ensemble\nBefore  Stacking Ensemble\nAfter  AdaBoost\nBefore  AdaBoost\nAfter  SVM (RBF)\nBefore  SVM (RBF)\nAfter  MLP\nBefore  MLP\nAfter
Metric                                                                                                                                                                                                                                                     
Accuracy                  0.9955                0.9879                  0.9886                 0.9826                     0.9955                    0.9902            0.3136           0.1765             0.9841            0.9871       0.9909      0.9909
Precision                 0.9957                0.9884                  0.9897                 0.9837                     0.9957                    0.9906            0.1766           0.1103             0.9856            0.9883       0.9915     

In [11]:
# ── Dark-theme matplotlib table ───────────────────────────────────────────────
n_methods = len(method_names)
n_cols    = n_methods * 2  # Before + After per method
n_rows    = len(metric_names)

fig, ax = plt.subplots(figsize=(n_cols * 1.5 + 1, n_rows * 0.7 + 1.5))
fig.patch.set_facecolor(P['bg'])
ax.set_facecolor(P['bg'])
ax.axis('off')

# Column headers: Method (Before | After)
col_labels = []
for m in method_names:
    short = m.split()[0]
    col_labels += [f'{short}\nBefore', f'{short}\nAfter']

cell_text  = []
cell_colors = []

for mname, mlabel in zip(metrics, metric_names):
    row_text   = []
    row_colors = []
    for name in method_names:
        before_val = results[name]['before'][mname]
        after_val  = results[name]['after'][mname]
        row_text.append(f'{before_val:.4f}')
        row_text.append(f'{after_val:.4f}')
        row_colors.append(P['panel'])
        # Green if After > Before, red if worse
        if after_val > before_val + 1e-6:
            row_colors.append(P['green'] + '55')  # semi-transparent
        elif after_val < before_val - 1e-6:
            row_colors.append(P['red'] + '55')
        else:
            row_colors.append(P['panel'])
    cell_text.append(row_text)
    cell_colors.append(row_colors)

table = ax.table(
    cellText=cell_text,
    rowLabels=metric_names,
    colLabels=col_labels,
    cellColours=cell_colors,
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor(P['border'])
    cell.set_text_props(color=P['text'])
    if row == 0 or col == -1:
        cell.set_facecolor(P['panel'])
        cell.set_text_props(color=P['text'], fontweight='bold')

ax.set_title('Synthesis Benchmark — 6 Methods × Before/After Augmentation',
             color=P['text'], fontsize=11, pad=12)

out = str(OUT_DIR / 'synthesis_table.png')
plt.tight_layout()
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved synthesis_table.png


In [12]:
# ── Grouped bar chart — Accuracy Before vs After ──────────────────────────────
x      = np.arange(n_methods)
width  = 0.35

acc_before = [results[n]['before']['accuracy'] for n in method_names]
acc_after  = [results[n]['after']['accuracy']  for n in method_names]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(P['bg'])
ax.set_facecolor(P['panel'])
ax.spines['bottom'].set_color(P['border'])
ax.spines['left'].set_color(P['border'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

bars_before = ax.bar(x - width/2, acc_before, width, label='Before augmentation',
                     color='none', edgecolor=P['blue'], linewidth=2)
bars_after  = ax.bar(x + width/2, acc_after,  width, label='After augmentation',
                     color=P['blue'], edgecolor=P['blue'], linewidth=2, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(method_names, rotation=15, ha='right', color=P['text'], fontsize=9)
ax.set_ylabel('Accuracy', color=P['text'])
ax.set_title('Classification Accuracy — Before vs After Multivariate Augmentation',
             color=P['text'], fontsize=11)
ax.tick_params(colors=P['text'])
ax.set_ylim(0, 1.05)
ax.yaxis.grid(True, color=P['border'], linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(facecolor=P['panel'], edgecolor=P['border'], labelcolor=P['text'])

for bar, val in zip(bars_before, acc_before):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', color=P['text'], fontsize=7)
for bar, val in zip(bars_after, acc_after):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', color=P['text'], fontsize=7)

out = str(OUT_DIR / 'synthesis_accuracy_chart.png')
plt.tight_layout()
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved synthesis_accuracy_chart.png


---
## Section 4 — Explainable AI

### 4.1 LIME — Local Interpretable Model-Agnostic Explanations

LIME approximates the model locally with a weighted linear model fitted to perturbed samples around the instance being explained.  
Here we explain a single rice prediction using the Random Forest trained on the augmented dataset.

### 4.2 SHAP — SHapley Additive exPlanations

SHAP assigns each feature a contribution score based on the Shapley value from cooperative game theory.  
`TreeExplainer` computes exact Shapley values for tree-based models in polynomial time.

In [13]:
import lime
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=X_tr_a,
    feature_names=FEATURES,
    class_names=list(rf_aug_pipeline.named_steps['clf'].classes_),
    mode='classification',
    random_state=42,
)

exp = lime_explainer.explain_instance(
    rice_sample,
    rf_aug_pipeline.predict_proba,
    num_features=7,
    top_labels=1,
)

print(f'Explained label : {list(exp.available_labels())}')
label_idx = exp.available_labels()[0]
print(f'Class name      : {rf_aug_pipeline.named_steps["clf"].classes_[label_idx]}')
print('Feature contributions:')
for feat, weight in exp.as_list(label=label_idx):
    print(f'  {feat:<40}: {weight:+.4f}')

Explained label : [np.int64(20)]
Class name      : rice
Feature contributions:
  rainfall > 124.38                       : +0.0621
  80.52 < humidity <= 89.90               : +0.0451
  31.46 < K <= 49.02                      : +0.0440
  temperature <= 22.71                    : +0.0138
  37.08 < N <= 84.28                      : +0.0097
  28.00 < P <= 50.54                      : +0.0094
  5.96 < ph <= 6.43                       : -0.0006


In [14]:
# Save LIME figure
fig = exp.as_pyplot_figure(label=label_idx)
fig.set_facecolor(P['bg'])
fig.axes[0].set_facecolor(P['panel'])
fig.axes[0].tick_params(colors=P['text'])
fig.axes[0].set_title(f'LIME Explanation — Rice Prediction (RF after augmentation)',
                      color=P['text'], fontsize=10)
out = str(OUT_DIR / 'lime_explanation.png')
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved lime_explanation.png


In [15]:
import shap

rf_clf   = rf_aug_pipeline.named_steps['clf']
mm_sc    = rf_aug_pipeline.named_steps['mm']
std_sc   = rf_aug_pipeline.named_steps['std']

# Scale the test set and the rice sample through the two scalers
X_te_scaled   = std_sc.transform(mm_sc.transform(X_te_a))
rice_scaled   = std_sc.transform(mm_sc.transform(rice_sample.reshape(1, -1)))

try:
    shap_explainer = shap.TreeExplainer(rf_clf)
    shap_values_all  = shap_explainer.shap_values(X_te_scaled)
    rice_shap_values = shap_explainer.shap_values(rice_scaled)
    print('TreeExplainer succeeded')
    use_tree = True
except Exception as e:
    print(f'TreeExplainer failed ({e}), falling back to KernelExplainer...')
    bg = shap.sample(X_te_scaled, 50)
    shap_explainer = shap.KernelExplainer(rf_clf.predict_proba, bg)
    shap_values_all  = shap_explainer.shap_values(X_te_scaled, check_additivity=False)
    rice_shap_values = shap_explainer.shap_values(rice_scaled, check_additivity=False)
    print('KernelExplainer succeeded')
    use_tree = False

# Locate rice class index
rice_class_idx = list(rf_clf.classes_).index('rice')
print(f'Rice class index: {rice_class_idx} (class: {rf_clf.classes_[rice_class_idx]})')

TreeExplainer succeeded
Rice class index: 20 (class: rice)


In [16]:
# ── SHAP waterfall for the rice sample ────────────────────────────────────────
if isinstance(shap_values_all, list):
    rice_sv   = rice_shap_values[rice_class_idx][0]
    base_val  = shap_explainer.expected_value[rice_class_idx]
else:
    rice_sv   = rice_shap_values[0, :, rice_class_idx]
    base_val  = shap_explainer.expected_value[rice_class_idx]

expl = shap.Explanation(
    values       = rice_sv,
    base_values  = base_val,
    data         = rice_scaled[0],
    feature_names= FEATURES,
)

plt.figure(figsize=(8, 4))
shap.plots.waterfall(expl, show=False)
plt.gcf().set_facecolor(P['bg'])
plt.title('SHAP Waterfall — Rice Sample (RF after augmentation)',
          color=P['text'], fontsize=10)
out = str(OUT_DIR / 'shap_waterfall_rice.png')
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved shap_waterfall_rice.png


In [17]:
# ── SHAP beeswarm summary across the full test set ────────────────────────────
if isinstance(shap_values_all, list):
    sv_all = shap_values_all[rice_class_idx]
else:
    sv_all = shap_values_all[:, :, rice_class_idx]

plt.figure(figsize=(8, 5))
shap.summary_plot(sv_all, X_te_scaled, feature_names=FEATURES, show=False)
plt.gcf().set_facecolor(P['bg'])
plt.title('SHAP Summary (Beeswarm) — Rice Class, All Test Samples',
          color=P['text'], fontsize=10)
out = str(OUT_DIR / 'shap_summary.png')
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved shap_summary.png


---
## Section 5 — LLM Architecture Diagram

Visualises the four-agent GPS-grounded crop recommendation pipeline.

In [18]:
fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor(P['bg'])
ax.set_facecolor(P['bg'])
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')

# Agent boxes: (x_center, color, title, subtitle)
agents = [
    (2.0,  P['blue'],   'Agent 1',  'Soil Analyzer',  'SoilGrids 2.0\n{N, P, K, ph}'),
    (5.0,  P['green'],  'Agent 2',  'Weather Analyst', 'Open-Meteo\n{temp, humidity, rain}'),
    (8.0,  P['amber'],  'Agent 3',  'Crop Predictor',  'ONNX Runtime\ncrop_model.onnx'),
    (11.0, P['purple'], 'Agent 4',  'Insight Engine',  'Groq LPU\nLlama 3.3 70B'),
]

box_w, box_h = 2.2, 2.0

for x, color, tag, name, desc in agents:
    # Shadow
    ax.add_patch(FancyBboxPatch(
        (x - box_w/2 + 0.05, 0.95), box_w, box_h,
        boxstyle='round,pad=0.1', linewidth=0,
        facecolor='#000000', alpha=0.4))
    # Box
    ax.add_patch(FancyBboxPatch(
        (x - box_w/2, 1.0), box_w, box_h,
        boxstyle='round,pad=0.1', linewidth=2,
        edgecolor=color, facecolor=P['panel']))
    # Top colour bar
    ax.add_patch(FancyBboxPatch(
        (x - box_w/2, 1.0 + box_h - 0.35), box_w, 0.35,
        boxstyle='round,pad=0.05', linewidth=0,
        facecolor=color, alpha=0.85))
    ax.text(x, 1.0 + box_h - 0.175, tag,
            ha='center', va='center', fontsize=7, fontweight='bold',
            color=P['bg'])
    ax.text(x, 1.0 + box_h*0.48, name,
            ha='center', va='center', fontsize=9, fontweight='bold',
            color=P['text'])
    ax.text(x, 1.0 + box_h*0.17, desc,
            ha='center', va='center', fontsize=7.5, color=P['sub'],
            linespacing=1.4)

# Arrows between agents
arrow_kw = dict(arrowstyle='->', color=P['text'], lw=1.5,
                mutation_scale=16, connectionstyle='arc3,rad=0')
arrow_y  = 2.0
for x_start, x_end, label in [
    (3.1,  3.9,  '{N,P,K,ph}'),
    (6.1,  6.9,  '{temp,hum,rain}'),
    (9.1,  9.9,  '{predicted_crop}'),
]:
    ax.annotate('', xy=(x_end, arrow_y), xytext=(x_start, arrow_y),
                arrowprops=arrow_kw)
    ax.text((x_start + x_end)/2, arrow_y + 0.22, label,
            ha='center', va='bottom', fontsize=6.5, color=P['sub'])

# Input / Output labels
ax.text(0.4, arrow_y, 'User\nlat,lon', ha='center', va='center',
        fontsize=8, color=P['text'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor=P['panel'],
                  edgecolor=P['border']))
ax.annotate('', xy=(0.9, arrow_y), xytext=(0.9, arrow_y),
            arrowprops=arrow_kw)
ax.annotate('', xy=(1.25, arrow_y), xytext=(0.75, arrow_y),
            arrowprops=arrow_kw)

ax.text(13.5, arrow_y, 'LLM\nReport', ha='center', va='center',
        fontsize=8, color=P['text'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor=P['panel'],
                  edgecolor=P['border']))
ax.annotate('', xy=(13.2, arrow_y), xytext=(12.1, arrow_y),
            arrowprops=arrow_kw)

ax.set_title('Crop AI — Four-Agent GPS-Grounded Pipeline',
             color=P['text'], fontsize=12, pad=8)

out = str(OUT_DIR / 'llm_architecture.png')
plt.tight_layout()
plt.savefig(out, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved {out}')

Saved llm_architecture.png


---
## Section 6 — Summary

In [19]:
# ── Generated output files ────────────────────────────────────────────────────
generated = [
    'synthesis_table.png',
    'synthesis_accuracy_chart.png',
    'lime_explanation.png',
    'shap_waterfall_rice.png',
    'shap_summary.png',
    'llm_architecture.png',
]

print('Generated files:')
for f in generated:
    exists = Path(f).exists()
    print(f'  {"OK" if exists else "MISSING":6}  {f}')

# ── Final accuracy summary (After augmentation) ───────────────────────────────
print('\nFinal Accuracy Summary (After Augmentation):')
print(f'{"Method":<22}  {"Before":>8}  {"After":>8}  {"Delta":>8}')
print('-' * 52)
for name in method_names:
    b = results[name]['before']['accuracy']
    a = results[name]['after']['accuracy']
    print(f'{name:<22}  {b:>8.4f}  {a:>8.4f}  {a-b:>+8.4f}')

Generated files:
  OK      synthesis_table.png
  OK      synthesis_accuracy_chart.png
  OK      lime_explanation.png
  OK      shap_waterfall_rice.png
  OK      shap_summary.png
  OK      llm_architecture.png

Final Accuracy Summary (After Augmentation):
Method                    Before     After     Delta
----------------------------------------------------
Random Forest             0.9955    0.9879   -0.0076
Gradient Boost            0.9886    0.9826   -0.0061
Stacking Ensemble         0.9955    0.9902   -0.0053
AdaBoost                  0.3136    0.1765   -0.1371
SVM (RBF)                 0.9841    0.9871   +0.0030
MLP                       0.9909    0.9909   +0.0000


---
## Section 7 — Deep Learning Extensions

Adds **TabNet** (attention-based tabular DL) and a **Hybrid DL+RF** (DenseNet feature extractor → Random Forest) to the synthesis benchmark.
Results are stored in the shared `results` dict using the same schema as the classical models.

In [ ]:
!pip install pytorch-tabnet torch -q
print('pytorch-tabnet and torch ready')

In [ ]:
# ── DL shared setup ──────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

PAPER_ASSETS = Path('..') / 'paper' / 'assets'
PAPER_ASSETS.mkdir(parents=True, exist_ok=True)

# Manual MinMax+Std preprocessing for DL models (same double-scaling as sklearn)
mm_dl   = MinMaxScaler()
std_dl  = StandardScaler()
le_dl   = LabelEncoder().fit(y_raw)          # integer label encoder

# Fit scalers on augmented train set (used for both TabNet and Hybrid)
X_tr_a_sc = std_dl.fit_transform(mm_dl.fit_transform(X_tr_a))
X_te_a_sc = std_dl.transform(mm_dl.transform(X_te_a))

# Separate scalers for raw (before-aug) split
mm_r  = MinMaxScaler()
std_r = StandardScaler()
X_tr_r_sc = std_r.fit_transform(mm_r.fit_transform(X_tr_r))
X_te_r_sc = std_r.transform(mm_r.transform(X_te_r))

y_tr_a_enc = le_dl.transform(y_tr_a)
y_te_a_enc = le_dl.transform(y_te_a)
y_tr_r_enc = le_dl.transform(y_tr_r)
y_te_r_enc = le_dl.transform(y_te_r)

print(f'Scaled shapes — aug train: {X_tr_a_sc.shape}, raw train: {X_tr_r_sc.shape}')
print(f'Classes ({len(le_dl.classes_)}): {list(le_dl.classes_[:5])} ...')


In [ ]:
# ── TabNet (Deep Learning model) ──────────────────────────────────────────────
from pytorch_tabnet.tab_model import TabNetClassifier

def train_tabnet(X_tr, y_tr_enc, X_te, y_te_enc, tag=''):
    clf = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5, gamma=1.3,
        n_independent=2, n_shared=2,
        momentum=0.02, epsilon=1e-15,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-3),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params=dict(step_size=50, gamma=0.9),
        mask_type='sparsemax', verbose=0
    )
    clf.fit(
        X_tr, y_tr_enc,
        eval_set=[(X_te, y_te_enc)],
        max_epochs=200, patience=20,
        batch_size=256, virtual_batch_size=128
    )
    y_pred_enc = clf.predict(X_te)
    y_pred     = le_dl.inverse_transform(y_pred_enc)
    y_te_str   = le_dl.inverse_transform(y_te_enc)
    acc  = accuracy_score(y_te_str, y_pred)
    prec = precision_score(y_te_str, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_te_str, y_pred, average='macro', zero_division=0)
    f1v  = f1_score(y_te_str, y_pred, average='macro', zero_division=0)
    print(f'  TabNet [{tag}]  Acc={acc:.4f}  F1={f1v:.4f}')
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1v, 'model': clf}

print('Training TabNet (before augmentation)...')
tn_before = train_tabnet(X_tr_r_sc, y_tr_r_enc, X_te_r_sc, y_te_r_enc, 'before')

print('Training TabNet (after augmentation)...')
tn_after  = train_tabnet(X_tr_a_sc, y_tr_a_enc, X_te_a_sc, y_te_a_enc, 'after')

results['TabNet'] = {'before': tn_before, 'after': tn_after}
if 'TabNet' not in method_names:
    method_names.append('TabNet')
print('TabNet stored in results dict.')


In [ ]:
# ── Hybrid DL+RF (DenseNet feature extractor → Random Forest) ─────────────────

class DenseExtractor(nn.Module):
    """7 → 64 → 64 → 32 MLP with BN + Dropout, outputs 32-dim embedding."""
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(7, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 64), nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
        )
        self.head = nn.Linear(32, 22)

    def embed(self, x):
        return self.enc(x)

    def forward(self, x):
        return self.head(self.embed(x))


def train_hybrid(X_tr, y_tr_enc, X_te, y_te_enc, tag='', epochs=150):
    device = torch.device('cpu')
    model  = DenseExtractor().to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    crit   = nn.CrossEntropyLoss()

    Xtr_t  = torch.tensor(X_tr, dtype=torch.float32)
    ytr_t  = torch.tensor(y_tr_enc, dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=256, shuffle=True)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        sched.step(epoch_loss)

    # Extract embeddings with frozen extractor
    model.eval()
    with torch.no_grad():
        emb_tr = model.embed(torch.tensor(X_tr, dtype=torch.float32)).numpy()
        emb_te = model.embed(torch.tensor(X_te, dtype=torch.float32)).numpy()

    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(emb_tr, y_tr_enc)
    y_pred_enc = rf.predict(emb_te)
    y_pred     = le_dl.inverse_transform(y_pred_enc)
    y_te_str   = le_dl.inverse_transform(y_te_enc)
    acc  = accuracy_score(y_te_str, y_pred)
    prec = precision_score(y_te_str, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_te_str, y_pred, average='macro', zero_division=0)
    f1v  = f1_score(y_te_str, y_pred, average='macro', zero_division=0)
    print(f'  Hybrid DL+RF [{tag}]  Acc={acc:.4f}  F1={f1v:.4f}')
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1v}


print('Training Hybrid DL+RF (before augmentation)...')
hyb_before = train_hybrid(X_tr_r_sc, y_tr_r_enc, X_te_r_sc, y_te_r_enc, 'before')

print('Training Hybrid DL+RF (after augmentation)...')
hyb_after  = train_hybrid(X_tr_a_sc, y_tr_a_enc, X_te_a_sc, y_te_a_enc, 'after')

results['Hybrid DL+RF'] = {'before': hyb_before, 'after': hyb_after}
if 'Hybrid DL+RF' not in method_names:
    method_names.append('Hybrid DL+RF')
print('Hybrid DL+RF stored in results dict.')


---
## Section 8 — Feature Analysis

Visualises the raw dataset distribution and inter-feature correlations. Both figures are saved directly to `paper/assets/` for inclusion in draft 3.

In [ ]:
# ── Feature Distribution Violin Plot ─────────────────────────────────────────
import seaborn as sns

feat_display = [
    ('N',           'N (mg/kg)'),
    ('P',           'P (mg/kg)'),
    ('K',           'K (mg/kg)'),
    ('temperature', 'Temperature (°C)'),
    ('humidity',    'Humidity (%)'),
    ('ph',          'pH'),
    ('rainfall',    'Rainfall (mm)'),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 12))
fig.patch.set_facecolor(P['bg'])
fig.suptitle('Feature Distributions Across 22 Crop Classes  (N = 2,200)',
             fontsize=13, color=P['text'], fontweight='bold', y=0.98)

axes_flat = axes.flatten()
for i, (feat, label) in enumerate(feat_display):
    ax = axes_flat[i]
    sns.violinplot(data=raw_df, x='label', y=feat, ax=ax,
                   color=P['blue'], linewidth=0.8, inner='quartile', alpha=0.75)
    ax.set_facecolor(P['panel'])
    ax.set_title(label, fontsize=10, color=P['text'], fontweight='bold', pad=6)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(colors=P['text'], labelsize=6.5)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=6.0)
    for spine in ax.spines.values():
        spine.set_color(P['border'])

axes_flat[7].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.96])

out_path = PAPER_ASSETS / 'feature_distribution.png'
fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved: {out_path}')


In [ ]:
# ── Feature Correlation Heatmap ───────────────────────────────────────────────
import seaborn as sns

display_names = ['N', 'P', 'K', 'Temp', 'Humidity', 'pH', 'Rainfall']
corr = raw_df[FEATURES].corr()
corr.index   = display_names
corr.columns = display_names

fig, ax = plt.subplots(figsize=(9, 7.5))
fig.patch.set_facecolor(P['bg'])
ax.set_facecolor(P['panel'])

sns.heatmap(
    corr, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.8, linecolor=P['border'],
    square=True,
    annot_kws={'size': 9, 'color': P['text']},
    ax=ax,
)
ax.set_title('Pearson Correlation Matrix — Agronomic Features',
             fontsize=11, color=P['text'], pad=12)
ax.tick_params(colors=P['text'], labelsize=9)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors=P['text'], labelsize=8)
cbar.set_label('Pearson r', color=P['text'], fontsize=9)

plt.tight_layout()
out_path = PAPER_ASSETS / 'feature_heatmap.png'
fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved: {out_path}')


---
## Section 9 — GradCAM++ Attribution (1D CNN)

Trains a lightweight 1D CNN on the tabular features (reshaped as a 7-element pseudo-sequence) and computes **GradCAM++** class activation maps. The spatial dimension of the last conv layer corresponds directly to the 7 feature positions, so each bar in the output plot is a per-feature attribution.

In [ ]:
# ── GradCAM++ on 1D CNN (tabular adaptation) ──────────────────────────────────

class CropCNN(nn.Module):
    """1D CNN: (batch,1,7) → 22 logits. relu2 = GradCAM++ hook target."""
    def __init__(self, n_classes=22):
        super().__init__()
        self.conv1 = nn.Conv1d(1,  32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()   # separate module → hookable post-ReLU feature maps
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.fc    = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.relu1(self.conv1(x))   # (batch, 32, 7)
        x = self.relu2(self.conv2(x))   # (batch, 64, 7) ← GradCAM++ target
        x = self.pool(x).squeeze(-1)    # (batch, 64)
        return self.fc(x)


# ── Train CNN ─────────────────────────────────────────────────────────────────
cnn   = CropCNN(n_classes=22)
opt_c = torch.optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
crit_c = nn.CrossEntropyLoss()

Xcnn = torch.tensor(X_tr_a_sc, dtype=torch.float32).unsqueeze(1)  # (N,1,7)
ycnn = torch.tensor(y_tr_a_enc, dtype=torch.long)
cnn_loader = DataLoader(TensorDataset(Xcnn, ycnn), batch_size=256, shuffle=True)

for epoch in range(100):
    cnn.train()
    for xb, yb in cnn_loader:
        opt_c.zero_grad()
        crit_c(cnn(xb), yb).backward()
        opt_c.step()

cnn.eval()
with torch.no_grad():
    Xte_cnn = torch.tensor(X_te_a_sc, dtype=torch.float32).unsqueeze(1)
    cnn_acc = (cnn(Xte_cnn).argmax(1) == torch.tensor(y_te_a_enc)).float().mean().item()
print(f'1D CNN test accuracy: {cnn_acc:.4f}  (visualization-only model)')


# ── GradCAM++ function ────────────────────────────────────────────────────────
def gradcam_pp(model, x_input, class_idx):
    """
    x_input : (1, 1, 7) float32 tensor
    Returns  : (7,) numpy array, normalised [0, 1]
    """
    activations, gradients = [None], [None]

    fwd_h = model.relu2.register_forward_hook(
        lambda m, i, o: activations.__setitem__(0, o))
    bwd_h = model.relu2.register_full_backward_hook(
        lambda m, gi, go: gradients.__setitem__(0, go[0]))

    model.zero_grad()
    x = x_input.detach().clone()
    scores = model(x)
    scores[0, class_idx].backward()

    fwd_h.remove()
    bwd_h.remove()

    A = activations[0].detach()   # (1, 64, 7)
    G = gradients[0].detach()     # (1, 64, 7)

    # GradCAM++ weights: global-avg-pool of gradients over spatial dim
    weights = G.mean(dim=-1, keepdim=True)     # (1, 64, 1)
    cam = (weights * A).sum(dim=1).squeeze()   # (7,) — one value per feature
    cam = torch.relu(cam)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.numpy()


# ── Compute GradCAM++ for 4 representative crops ─────────────────────────────
Xte_cnn_np = X_te_a_sc
crop_targets = ['rice', 'wheat', 'mango', 'cotton']
crop_cams    = {}

for crop in crop_targets:
    enc_idx = int(le_dl.transform([crop])[0])
    # Find test sample where model predicts this crop
    with torch.no_grad():
        preds = cnn(Xte_cnn).argmax(1).numpy()
    match = np.where(preds == enc_idx)[0]
    if len(match) == 0:                        # fallback: true label
        match = np.where(y_te_a_enc == enc_idx)[0]
    sample = Xte_cnn[match[0]].unsqueeze(0)
    cam = gradcam_pp(cnn, sample, enc_idx)
    crop_cams[crop] = cam
    top_feat = ['N','P','K','Temp','Humidity','pH','Rainfall'][cam.argmax()]
    print(f"  GradCAM++ '{crop}': top feature = {top_feat}  ({cam.max():.3f})")


# ── Plot 2×2 grid ─────────────────────────────────────────────────────────────
feat_labels = ['N', 'P', 'K', 'Temp', 'Humidity', 'pH', 'Rainfall']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor(P['bg'])
fig.suptitle('GradCAM++ Attribution Analysis — 1D CNN (Last Convolutional Layer)',
             fontsize=11, color=P['text'], fontweight='bold')

for ax, crop in zip(axes.flatten(), crop_targets):
    cam = crop_cams[crop]
    top_i = cam.argmax()
    bar_colors = [P['amber'] if j == top_i else P['blue'] for j in range(7)]
    bars = ax.barh(feat_labels, cam, color=bar_colors,
                   edgecolor=P['border'], linewidth=0.5)
    for bar, val in zip(bars, cam):
        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}', va='center', ha='left', fontsize=8, color=P['text'])
    ax.set_facecolor(P['panel'])
    ax.set_title(f'GradCAM++ — {crop.capitalize()}',
                 fontsize=10, color=P['text'], fontweight='bold')
    ax.set_xlabel('GradCAM++ Activation (normalised)', fontsize=8, color=P['sub'])
    ax.set_xlim(0, 1.2)
    ax.tick_params(colors=P['text'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(P['border'])

plt.tight_layout()
out_path = PAPER_ASSETS / 'gradcam_dl.png'
fig.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved: {out_path}')


---
## Section 10 — Extended Synthesis (8 Methods)

Re-generates the synthesis table and accuracy bar chart with all 8 methods (original 6 + TabNet + Hybrid DL+RF).

In [ ]:
# ── Extended Synthesis Table — 8 Methods ─────────────────────────────────────
metrics_keys  = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1']

n_m8   = len(method_names)
n_c8   = n_m8 * 2
n_r8   = len(metric_labels)

rows8 = []
for mkey, mlabel in zip(metrics_keys, metric_labels):
    row = {'Metric': mlabel}
    for name in method_names:
        row[f'{name}\nBefore'] = round(results[name]['before'][mkey], 4)
        row[f'{name}\nAfter']  = round(results[name]['after'][mkey],  4)
    rows8.append(row)
df8 = pd.DataFrame(rows8)

fig8, ax8 = plt.subplots(figsize=(n_c8 * 1.5 + 1, n_r8 * 0.7 + 1.5))
fig8.patch.set_facecolor(P['bg'])
ax8.set_facecolor(P['bg'])
ax8.axis('off')

col_labels8 = []
for m in method_names:
    short = m.split()[0]
    col_labels8 += [f'{short}\nBefore', f'{short}\nAfter']

cell_text8 = [[row['Metric']] + [df8[c].iloc[i] for c in df8.columns[1:]]
              for i, row in df8.iterrows()]

tbl8 = ax8.table(
    cellText=cell_text8,
    colLabels=['Metric'] + col_labels8,
    loc='center', cellLoc='center',
)
tbl8.auto_set_font_size(False)
tbl8.set_fontsize(8)
tbl8.scale(1, 1.6)

for j in range(n_c8 + 1):
    cell = tbl8[0, j]
    cell.set_facecolor(P['panel'])
    cell.set_text_props(color=P['text'], fontweight='bold')
    cell.set_edgecolor(P['border'])

for i in range(1, n_r8 + 1):
    for j in range(n_c8 + 1):
        cell = tbl8[i, j]
        cell.set_facecolor(P['bg'])
        cell.set_edgecolor(P['border'])
        if j == 0:
            cell.set_text_props(color=P['sub'], fontsize=8)
        else:
            try:
                v = float(str(cell.get_text().get_text()))
                col = P['green'] if v >= 0.99 else (
                      P['blue']  if v >= 0.95 else (
                      P['amber'] if v >= 0.90 else P['text']))
            except ValueError:
                col = P['text']
            cell.set_text_props(color=col, fontsize=8)

ax8.set_title('Synthesis Benchmark — 8 Methods × Before / After Augmentation',
              color=P['text'], fontsize=11, fontweight='bold', pad=16)
plt.tight_layout()

out_path = PAPER_ASSETS / 'synthesis_table_v2.png'
fig8.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved: {out_path}')


In [ ]:
# ── Accuracy Bar Chart — 8 Methods ───────────────────────────────────────────
x8    = np.arange(len(method_names))
w8    = 0.30

acc_b8 = [results[n]['before']['accuracy'] for n in method_names]
acc_a8 = [results[n]['after']['accuracy']  for n in method_names]

fig9, ax9 = plt.subplots(figsize=(14, 5))
fig9.patch.set_facecolor(P['bg'])
ax9.set_facecolor(P['panel'])
ax9.spines['bottom'].set_color(P['border'])
ax9.spines['left'].set_color(P['border'])
ax9.spines['top'].set_visible(False)
ax9.spines['right'].set_visible(False)

ax9.bar(x8 - w8/2, acc_b8, w8, label='Before Aug', color=P['blue'],  alpha=0.85)
ax9.bar(x8 + w8/2, acc_a8, w8, label='After Aug',  color=P['green'], alpha=0.85)

short8 = []
for n in method_names:
    if 'DL+RF' in n:
        short8.append('Hybrid\nDL+RF')
    else:
        short8.append(n.split()[0])

ax9.set_xticks(x8)
ax9.set_xticklabels(short8, color=P['text'], fontsize=8, rotation=15, ha='right')
ax9.set_ylabel('Accuracy', color=P['text'], fontsize=10)
ax9.set_ylim(0.4, 1.05)
ax9.tick_params(colors=P['text'])
ax9.set_title('Classification Accuracy — 8-Method Benchmark (Before / After Augmentation)',
              color=P['text'], fontsize=11, pad=10)
ax9.legend(facecolor=P['panel'], edgecolor=P['border'],
           labelcolor=P['text'], fontsize=9)

plt.tight_layout()
out_path = PAPER_ASSETS / 'synthesis_accuracy_chart_v2.png'
fig9.savefig(out_path, dpi=130, bbox_inches='tight', facecolor=P['bg'])
plt.close()
print(f'Saved: {out_path}')


In [ ]:
# ── Generated output files (draft 3) ─────────────────────────────────────────
legacy_files = [
    OUT_DIR / 'synthesis_table.png',
    OUT_DIR / 'synthesis_accuracy_chart.png',
    OUT_DIR / 'lime_explanation.png',
    OUT_DIR / 'shap_waterfall_rice.png',
    OUT_DIR / 'shap_summary.png',
    OUT_DIR / 'llm_architecture.png',
]
new_files = [
    PAPER_ASSETS / 'feature_distribution.png',
    PAPER_ASSETS / 'feature_heatmap.png',
    PAPER_ASSETS / 'gradcam_dl.png',
    PAPER_ASSETS / 'synthesis_table_v2.png',
    PAPER_ASSETS / 'synthesis_accuracy_chart_v2.png',
]

print('=== Original figures ===')
for p in legacy_files:
    print(f'  {"OK" if p.exists() else "MISSING":6}  {p}')

print()
print('=== New figures (draft 3) ===')
for p in new_files:
    print(f'  {"OK" if p.exists() else "MISSING":6}  {p}')

print()
print('=== Final accuracy (after augmentation) — all 8 methods ===')
for name in method_names:
    acc = results[name]['after']['accuracy']
    f1v = results[name]['after']['f1']
    print(f'  {name:<22}  acc={acc:.4f}  f1={f1v:.4f}')
